# Learning Rate Schedule

**Llama 3 Pre-training:** 
- "We pre-train **Llama 3 405B** using **AdamW** with a **peak learning rate of** $\mathbf{8 × 10^{−5}}$ , **a linear warm up of 8,000 steps**, and **a cosine learning rate schedule decaying to** $\mathbf{8 × 10^{−7}}$ **over** $\mathbf{1{,}200{,}000}$ **steps**." 
- During pre-training on the final $40\text{M}$ tokens, we **linearly annealed the learning rate** to $0$

**Llama 2 paper:**
- Hyperparameters. We trained using the **AdamW** optimizer (Loshchilov and Hutter, 2017), with $\boldsymbol{\beta_1 = 0.9}$, $\boldsymbol{\beta_2 = 0.95}$, $\cancel{eps = 10^{−5}}$. ~~We use a cosine learning rate schedule~~, ~~with warmup of~~ $\cancel{2000}$ ~~steps, and decay final learning rate down to 10% of the peak learning rate**. We use a weight decay of $\cancel{0.1}$ and gradient clipping of 1.0. Figure 5 (a) shows the training loss for Llama 2 with these hyperparameters.~~

*The items like $\beta_1$ aren't mentioned in the third paper, so they are referenced in the second paper*

**Note:** In the Llama 3 paper, specifically when they are pre-training the encoders for the **multi-modal Vision** they mentioned "We use a global batch size of $16{,}384$ and a cosine learning rate schedule with initial learning rate $10 × 10^{−4}$ and a weight decay of $0.1$. The initial learning rate was determined based on small-scale experiments."
#TODO is weight decay 0.1 or 0.01? here: weight decay of $0.1$???

In [1]:
import easyjupyter
import torch
import torch.nn as nn

In [ ]:
import math
import torch.optim as optim
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from configs import Llama3_1_405B #TODO change to scaled down

In [3]:
def get_lr_multiplier(cfg, current_step: int):
    """
    Calculates the learning rate multiplier for the Llama 3 schedule.
        1. Linear warmup from 0 to peak_lr.
        2. Cosine decay from peak_lr down to min_lr_ratio, ending exactly when annealing starts.
        3. Linear decay from min_lr_ratio down to 0 over the final tokens, during the annealing stage of pre-training.
    """

    # Calculate when annealing begins
    annealing_perc = cfg.pre_training_stages_steps["annealing_perc"]
    annealing_steps = int(cfg.total_training_steps * annealing_perc)
    annealing_start_step = cfg.total_training_steps - annealing_steps
    min_lr_ratio = getattr(cfg, "min_lr_ratio", 0.1)

    # Annealing sub-stage, linear decay from min_lr_ratio to 0.
    if current_step >= annealing_start_step:
        steps_into_annealing = current_step - annealing_start_step
        # linear decay the remaining multiplier to 0
        decay_factor = 1.0 - (steps_into_annealing / max(1, annealing_steps))
        return min_lr_ratio * max(0.0, decay_factor)

    # Linear warmup phase in the initial sub-stage
    if current_step < cfg.warmup_steps:
        # return a fraction from near 0.0 up to 1.0
        return float(current_step) / float(max(1, cfg.warmup_steps))

    # Cosine Decay Phase: Calculate progress from 0.0 (end of warmup) to 1.0 (end of token budget)
    decay_progress = float(current_step - cfg.warmup_steps) / float(
        max(1, annealing_start_step - cfg.warmup_steps)
    )

    # Cap the step so the learning rate doesn't start rising again
    decay_progress = min(1.0, decay_progress)

    # Cosine: scales down from 1.0 to 0.0
    cosine_decay = 0.5 * (1.0 + math.cos(math.pi * decay_progress))

    # Scale it to the stop at min_lr_ratio (10% or 1% depending on config) instead of hitting 0
    return min_lr_ratio + (1.0 - min_lr_ratio) * cosine_decay


def get_scheduler(
    cfg: BaseConfig, optimizer: torch.optim.AdamW, isAnnealing: bool = False
):
    """
    The scheduler will be different depending on the training phase configurations from cfg.

    Args:
        isAnnealing: True if we are at the annealing stage of pre-training, where we decayed the learning rate to $0$ past the `min_lr_ratio`, change the data mix to the absolute highest quality math and reasoning data sources.
    """
    return torch.optim.lr_scheduler.LambdaLR(
        optimizer=optimizer,
        lr_lambda=lambda step: get_lr_multiplier(
            current_step=step,
            cfg=cfg,
        ),
    )


def get_optimizer(cfg: BaseConfig, model: nn.Module):
    """ """
    return optim.AdamW(
        model.parameters(),
        lr=cfg.peak_lr,
        betas=(0.9, 0.95),
        eps=1e-8,
        weight_decay=0.1,  # TODO verify params in paper
    )

## Test

In [54]:
# @i-c
print("\n------ LR Schedule Pre-Training Example For The Llama 3 405B Model ------\n")
from llama_configs import Llama3_scaled_down
import math

cfg = Llama3_scaled_down()
cfg.warmup_steps = 8_000
cfg.peak_lr = 8e-5
cfg.token_budget = 1_200_000 * cfg.global_batch_size_tokens
total_tokens = 15_600_000_000_000
annealing_tokens = 40_000_000
lc_tokens = 800_000_000_000
initial_tokens = total_tokens - lc_tokens - annealing_tokens

cfg.pre_training_stages_steps = {
    "initial_perc": initial_tokens / total_tokens,
    "long_context_perc": lc_tokens / total_tokens,
    "annealing_perc": annealing_tokens / total_tokens,
}


print(f"\n\nTotal training steps: {cfg.total_training_steps:,}")
print(f"Total tokens: {total_tokens:,}")
print(f"Warmup ends at step: {cfg.warmup_steps:,}")
print(
    f"Peak learning rate: {cfg.peak_lr:.8f} | Min learning rate: {cfg.peak_lr * cfg.min_lr_ratio:.8f}"
)
print(f"\n\n{'Step':>9} | {'Learning-Rate':>15} | {'Tokens Seen':>18} | {'Phase'}")
print("-" * 100)


# Calculate when annealing begins
annealing_perc = cfg.pre_training_stages_steps["annealing_perc"]
annealing_steps = max(1, int(cfg.total_training_steps * annealing_perc))
anneal_start_step = cfg.total_training_steps - annealing_steps

milestones = [  # Training milestones
    1,                          # Starts warmup
    4_000,                      # Halfway through warmup
    8_000,                      # Warmup ends (Peak LR)
    100_000,                    # Early lr decay
    600_000,                    # Mid lr decay
    1_000_000,                  # late lr decay
    anneal_start_step - 1,      # The absolute last step of cosine decay
    anneal_start_step,          # annealing starts (snaps to min lr)
    anneal_start_step + 1,      # mid annealing lr approaches 0
    cfg.total_training_steps,   # End of training (lr hits 0)
]

for step in milestones:
    multiplier = get_lr_multiplier(current_step=step, cfg=cfg)
    lr = cfg.peak_lr * multiplier

    if step <= cfg.warmup_steps:
        phase = "Warmup (Linear up ↑)"
    elif step < anneal_start_step:
        phase = "Decay (Cosine down ↓)"
    elif step < cfg.total_training_steps:
        phase = "Annealing (Linear down to 0 ↓)"
    else:
        phase = "Training complete (Flat at 0)"

    tokens_per_step = total_tokens / cfg.total_training_steps
    tokens_seen = int(step * tokens_per_step)
    print(f"{step:08,d} | {lr:15.8f} | {tokens_seen:018,d} | {phase}")


------ LR Schedule Pre-Training Example For The Llama 3 405B Model ------


Project Root: /Users/tonyavis/Main/How_to_build_an_LLM


Total training steps: 1,200,000
Total tokens: 15,600,000,000,000
Warmup ends at step: 8,000
Peak learning rate: 0.00008000 | Min learning rate: 0.00000800


     Step |   Learning-Rate |        Tokens Seen | Phase
----------------------------------------------------------------------------------------------------
0,000,001 |      0.00000001 | 00,000,013,000,000 | Warmup (Linear up ↑)
0,004,000 |      0.00004000 | 00,052,000,000,000 | Warmup (Linear up ↑)
0,008,000 |      0.00008000 | 00,104,000,000,000 | Warmup (Linear up ↑)
0,100,000 |      0.00007895 | 01,300,000,000,000 | Decay (Cosine down ↓)
0,600,000 |      0.00004438 | 07,800,000,000,000 | Decay (Cosine down ↓)
1,000,000 |      0.00001289 | 13,000,000,000,000 | Decay (Cosine down ↓)
1,199,996 |      0.00000800 | 15,599,948,000,000 | Decay (Cosine down ↓)
1,199,997 |      0.00000800 | 15,599,961,00